# ML-07 — Baseline Action Score

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane: Growth / Recovery / Momentum Prediction** (confirmed — same lane and same March 2026 first-half/second-half split as `w03_data_contract.ipynb`; column names below are already verified against the real schema, carried over from that notebook's run).

> Skills loaded: `building-baselines/SKILL.md`, `flyrank/flyrank-data/SKILL.md` (per `skills/README.md`).

## 1. My rule, its reasoning, and two signal checks

**The rule, in plain words:** *A page is worth reviewing for its CTR if it already has real search visibility, but its click-through rate is below what pages at its own position typically get.* This is the same idea behind FlyRank's real `ctr_review_candidate` flag (impressions ≥ 500, position 1–20, CTR < 0.5) — rebuilt here from scratch on my lane's slice, not copied as a feature.

Before coding it, I check the two signals it leans on:

**Signal 1 (flag-linked — CTR-vs-position, behind FlyRank's real CTR-fix logic):** "Pages at a better (lower) GSC position get a higher click-through rate." Tested as **weighted CTR per position bucket** (`SUM(clicks) / SUM(impressions)` per bucket, not an average of per-page rates — averaging per-row rates isn't the true rate).

**Signal 2 (volume, behind the quick-win logic):** "Pages with more impression volume are less likely to be the ones declining next half." Tested as **decline rate per impression-volume bucket**, with n printed per bucket so no verdict rests on a tiny cell (floor: 50 rows).

In [ ]:
# ---- Setup: same DuckDB + HF pattern as w03, columns already verified there ----
%pip -q install duckdb
import duckdb
import pandas as pd
import numpy as np
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HF_TOKEN (plain Read token): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # same mid-panel month as w03 -- never the _sample (sealed final) month
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Confirmed real column names (verified against DESCRIBE output in w03_data_contract.ipynb)
COLS = {
    "impressions": "gsc_impressions",
    "clicks": "gsc_clicks",
    "position": "gsc_avg_position",
    "ga4_flag": "ga4_data_available",
}
print("Using verified columns:", COLS)


In [ ]:
# ---- Rebuild the same first-half feature frame as w03 (no future-window inputs) ----
# Feature window: Mar 1-15 (already happened, safe). Label window: Mar 16-31 (used ONLY to
# test signals and to score the rule afterward -- never as an input to the rule's score itself).
feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM({COLS['impressions']}) AS impressions_first_half,
            SUM({COLS['clicks']})      AS clicks_first_half,
            AVG(CASE WHEN {COLS['impressions']} > 0 THEN {COLS['position']} END) AS avg_position_first_half,
            COUNT(DISTINCT CASE WHEN {COLS['impressions']} > 0 THEN report_date END) AS active_days_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM({COLS['impressions']}) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id, f.content_hash_id,
        f.impressions_first_half,
        f.clicks_first_half,
        ROUND(100.0 * f.clicks_first_half / NULLIF(f.impressions_first_half, 0), 2) AS ctr_first_half,
        f.avg_position_first_half,
        f.active_days_first_half,
        COALESCE(s.impressions_second_half, 0) AS impressions_second_half,
        CASE
            WHEN f.impressions_first_half > 0
                 AND (COALESCE(s.impressions_second_half, 0) - f.impressions_first_half)
                     / f.impressions_first_half < -0.20
            THEN 1 ELSE 0
        END AS is_declining_next_half
    FROM first_half f
    LEFT JOIN second_half s USING (client_hash_id, content_hash_id)
    WHERE f.impressions_first_half > 0
""").df()

print(f"Rows: {len(feature_frame):,} (client, content) pairs")
feature_frame.head()


In [ ]:
# ---- Signal 1: CTR vs position bucket (weighted CTR, not averaged per-row rates) ----
def position_tier(p):
    if pd.isna(p) or p <= 0:
        return "no_data"
    if p <= 3:
        return "top_3"
    if p <= 10:
        return "page_1"
    if p <= 20:
        return "striking"
    if p <= 50:
        return "page_3_5"
    return "deep"

feature_frame["position_tier"] = feature_frame["avg_position_first_half"].apply(position_tier)

signal1 = (
    feature_frame
    .groupby("position_tier")
    .agg(n=("content_hash_id", "size"),
         total_impressions=("impressions_first_half", "sum"),
         total_clicks=("clicks_first_half", "sum"))
)
signal1["weighted_ctr_pct"] = (100 * signal1["total_clicks"] / signal1["total_impressions"]).round(2)
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
signal1 = signal1.reindex(tier_order)
print(signal1[["n", "weighted_ctr_pct"]])

floor_ok = signal1.loc[["top_3", "page_1", "striking", "page_3_5", "deep"], "n"].min() >= 50
monotonic = signal1.loc[["top_3", "page_1", "striking", "page_3_5", "deep"], "weighted_ctr_pct"].is_monotonic_decreasing
verdict1 = "CONFIRMED" if (floor_ok and monotonic) else ("MIXED" if floor_ok else "insufficient data")
print(f"\nSample-size floor met (n>=50 per bucket): {floor_ok}")
print(f"VERDICT (Signal 1, CTR-vs-position, flag-linked): {verdict1}")


In [ ]:
# ---- Signal 2: impression-volume bucket vs decline rate ----
def volume_tier(imp):
    if imp < 10:
        return "low (<10)"
    if imp < 100:
        return "moderate (10-99)"
    if imp < 1000:
        return "good (100-999)"
    return "high (1000+)"

feature_frame["volume_tier"] = feature_frame["impressions_first_half"].apply(volume_tier)

signal2 = (
    feature_frame
    .groupby("volume_tier")
    .agg(n=("content_hash_id", "size"),
         decline_rate_pct=("is_declining_next_half", lambda x: round(100 * x.mean(), 1)))
)
vol_order = ["low (<10)", "moderate (10-99)", "good (100-999)", "high (1000+)"]
signal2 = signal2.reindex(vol_order)
print(signal2)

floor_ok2 = signal2["n"].min() >= 50
decreasing = signal2["decline_rate_pct"].is_monotonic_decreasing
if not floor_ok2:
    verdict2 = "insufficient data"
elif decreasing:
    verdict2 = "CONFIRMED"
elif signal2["decline_rate_pct"].is_monotonic_increasing:
    verdict2 = "OPPOSITE"
else:
    verdict2 = "MIXED"
print(f"\nSample-size floor met (n>=50 per bucket): {floor_ok2}")
print(f"VERDICT (Signal 2, volume-vs-decline, quick-win-linked): {verdict2}")
print("A MIXED or OPPOSITE verdict here is a real, useful finding -- it means the rule below")
print("should NOT lean on volume alone as a stability signal.")


## 2. Build the ranked queue (writes the CSV)

**The rule, coded exactly as described above** — transparent, no fitted weights:

```text
has_volume     = impressions_first_half >= 50
low_ctr        = ctr_first_half < weighted_ctr_pct of its OWN position_tier bucket
score          = has_volume * low_ctr * impressions_first_half
reason_code    = "ctr_below_position_expectation"
action         = "review_ctr_snippet"
```

Only `impressions_first_half`, `clicks_first_half`, `ctr_first_half`, and `avg_position_first_half` go into the score — all Mar 1–15, all already-observed at the decision point. `impressions_second_half` / `is_declining_next_half` are used **only afterward**, to report precision@10 against the base rate — never as score inputs.

In [ ]:
# ---- Encode the rule ----
expected_ctr_by_tier = signal1["weighted_ctr_pct"].to_dict()
feature_frame["expected_ctr_for_tier"] = feature_frame["position_tier"].map(expected_ctr_by_tier)

MIN_IMPRESSIONS = 50
feature_frame["has_volume"] = (feature_frame["impressions_first_half"] >= MIN_IMPRESSIONS).astype(int)
feature_frame["low_ctr"] = (
    feature_frame["ctr_first_half"] < feature_frame["expected_ctr_for_tier"]
).astype(int)

feature_frame["score"] = (
    feature_frame["has_volume"] * feature_frame["low_ctr"] * feature_frame["impressions_first_half"]
)
feature_frame["reason_code"] = np.where(
    feature_frame["score"] > 0, "ctr_below_position_expectation", "none"
)
feature_frame["action"] = np.where(
    feature_frame["score"] > 0, "review_ctr_snippet", "no_action"
)

ranked = feature_frame.sort_values("score", ascending=False).reset_index(drop=True)
print(f"Rows flagged with score > 0: {(ranked['score'] > 0).sum():,} of {len(ranked):,}")
ranked.head(10)[["client_hash_id", "content_hash_id", "score", "reason_code", "action"]]


In [ ]:
# ---- Precision@10 vs base rate (evaluation only -- label was NOT a score input) ----
def precision_at_k(labels, k):
    return labels.head(k).mean()

base_rate = ranked["is_declining_next_half"].mean()
p_at_10 = precision_at_k(ranked["is_declining_next_half"], 10)
print(f"Base rate of is_declining_next_half across the whole slice: {base_rate:.1%}")
print(f"Precision@10 (top 10 by rule score): {p_at_10:.1%}")


In [ ]:
# ---- Write the ranked queue CSV (gitignored by design -- regenerated on every run) ----
os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "client_hash_id", "content_hash_id",
    "impressions_first_half", "clicks_first_half", "ctr_first_half",
    "avg_position_first_half", "position_tier", "expected_ctr_for_tier",
    "active_days_first_half", "score", "reason_code", "action",
]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv -- {len(ranked):,} rows")

# ---- Metrics receipt -- this one IS committed (small JSON, not raw data) ----
import json as _json
metrics = {
    "month": MONTH,
    "n_rows": int(len(ranked)),
    "n_flagged": int((ranked["score"] > 0).sum()),
    "base_rate_is_declining_next_half": round(float(base_rate), 4),
    "precision_at_10": round(float(p_at_10), 4),
    "signal1_ctr_vs_position_verdict": verdict1,
    "signal2_volume_vs_decline_verdict": verdict2,
    "min_impressions_threshold": MIN_IMPRESSIONS,
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)
print(metrics)


## 3. Top-10 review

For each of the top 10, one line each: the action, why it's there, and what would make it wrong — grounded in that row's own numbers (a low `active_days_first_half` or a near-boundary `avg_position_first_half` means the CTR estimate is noisier and the pick is weaker).

In [ ]:
top10 = ranked.head(10).copy()

def why_here(row):
    return (f"impressions_first_half={row.impressions_first_half:.0f}, "
            f"ctr_first_half={row.ctr_first_half:.2f}% vs expected "
            f"{row.expected_ctr_for_tier:.2f}% for its '{row.position_tier}' bucket "
            f"(position {row.avg_position_first_half:.1f})")

def what_would_make_it_wrong(row):
    notes = []
    if row.active_days_first_half < 5:
        notes.append(f"only {int(row.active_days_first_half)} active day(s) in the feature window -- CTR could be one lucky/unlucky day, not a real pattern")
    if row.impressions_first_half < 100:
        notes.append("volume is close to the 50-impression floor -- a few more impressions could move the CTR estimate a lot")
    if not notes:
        notes.append("looks solid on volume and day-coverage -- weakest risk is that position itself may shift next period")
    return "; ".join(notes)

top10["why_here"] = top10.apply(why_here, axis=1)
top10["what_would_make_it_wrong"] = top10.apply(what_would_make_it_wrong, axis=1)

for i, row in top10.iterrows():
    print(f"{i+1}. [{row.action}] {row.content_hash_id[:20]}...")
    print(f"   Why: {row.why_here}")
    print(f"   Could be wrong if: {row.what_would_make_it_wrong}")
    print()


## 4. Weak picks + leakage check

**Weakest picks:** rows in the top 10 flagged above with `active_days_first_half` under 5 are the weakest — a CTR built from 1–4 days is a noisy estimate, not a stable pattern, so those picks are more likely to be wrong than the rest of the list.

**Leakage check:** the score formula uses only `impressions_first_half`, `clicks_first_half`, `ctr_first_half`, and `avg_position_first_half` (via `position_tier`) — all Mar 1–15. `impressions_second_half` and `is_declining_next_half` were used only in the precision@10 printout after the queue was already ranked, never as score inputs. No FlyRank product flags (`health_score`, `priority_score`, etc.) exist in this dataset to accidentally use.

In [ ]:
# ---- Explicit, checkable leakage guard: assert the score never touched label-side columns ----
score_inputs = {"impressions_first_half", "clicks_first_half", "ctr_first_half",
                 "avg_position_first_half", "position_tier", "expected_ctr_for_tier", "has_volume", "low_ctr"}
label_side_columns = {"impressions_second_half", "is_declining_next_half"}

assert score_inputs.isdisjoint(label_side_columns), "Leakage: a label-side column was used in scoring!"
print("Confirmed: score inputs and label-side columns do not overlap.")
print("Score inputs used:", sorted(score_inputs))
print("Label-side columns (evaluation only):", sorted(label_side_columns))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.